# ModelOptions

`pyneat.ModelOptions` is the contract you give Neat before constructing a `pyneat.Model`.
It answers three practical questions:

- What kind of input will the application send?
- What preprocessing should Neat plan before inference?
- Should Neat decode model outputs, such as YOLO boxes, pose, or segmentation?

This notebook focuses only on `ModelOptions`. Runtime queueing and push/pull behavior belong to `RunOptions` and graph execution notebooks.


## Imports

Run this notebook from the DevKit pyneat environment.


In [1]:
from pathlib import Path
import numpy as np
import cv2
import pyneat


## Complete ModelOptions Field Map

This section lists the public `pyneat.ModelOptions` fields first, then expands every nested option group that has sub-options. Use it as a checklist before configuring a model.

### Root ModelOptions Fields

| Field | Type / values | One-line use |
| --- | --- | --- |
| `preprocess` | `PreprocessOptions` | Describes input type and model-managed preprocessing before inference. |
| `decode_type` | `BoxDecodeType` | Selects detection, pose, or segmentation decode family; leave `Unspecified` for raw outputs. |
| `decode_type_option` | `BoxDecodeTypeOption` | Selects tensor packing variant inside a decode family; keep `Auto` unless head layout is known. |
| `score_threshold` | `float` | Drops decoded candidates below this confidence score. |
| `nms_iou_threshold` | `float` | Controls non-max suppression overlap; `0` disables NMS. |
| `top_k` | `int` | Caps decoded results per frame/output for predictable downstream cost. |
| `num_classes` | `int` | Supplies class count when MPK metadata cannot infer it reliably. |
| `boxdecode_original_width` | `int` | Deprecated original-width hint for older boxdecode coordinate inversion paths. |
| `boxdecode_original_height` | `int` | Deprecated original-height hint for older boxdecode coordinate inversion paths. |
| `boxdecode_resize_mode` | `ResizeMode` or `None` | Optional legacy resize-mode hint for boxdecode coordinate inversion. |
| `upstream_name` | `str` | Names the upstream graph node feeding model preprocessing. |
| `name_suffix` | `str` | Appends a suffix to generated route element names to avoid name collisions. |
| `cleanup_extracted_model_data` | `bool` | Removes extracted model package files on exit when `True`; keep files for debugging with `False`. |
| `verbose` | `VerboseOptions` | Controls model construction, planner, graph, plugin, and tensor diagnostic output. |
| `inference_terminal` | `InferenceTerminalPolicy` | Advanced debug control for stopping the model route at a selected internal stage. |
| `processcvu` | `ProcessCvuOptions` | Low-level placement/async controls for model-managed CVU pre/post stages. |
| `processmla` | `ProcessMlaOptions` | Low-level async and buffering controls for model-managed MLA inference. |
| `advanced_execution` | `AdvancedExecutionOptions` | Preferred high-level execution tuning surface; use only after measuring defaults. |

### `preprocess` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `preprocess.kind` | `InputKind.Auto`, `Image`, `Tensor` | Declares whether the app sends decoded image pixels or already-shaped tensors. |
| `preprocess.enable` | `AutoFlag.Auto`, `On`, `Off` | Master switch for model-managed preprocessing. |
| `preprocess.input_max_width` | `int` | Maximum input image width accepted by the route; `0` means no explicit bound. |
| `preprocess.input_max_height` | `int` | Maximum input image height accepted by the route; `0` means no explicit bound. |
| `preprocess.input_max_depth` | `int` | Maximum input channel depth accepted by the route; `0` means no explicit bound. |
| `preprocess.resize` | `ResizeSpec` | Resize, letterbox, crop, padding, and interpolation settings. |
| `preprocess.color_convert` | `ColorConvertSpec` | Source and target image color format settings. |
| `preprocess.layout_convert` | `LayoutConvertSpec` | Axis-order conversion such as HWC to CHW. |
| `preprocess.normalize` | `NormalizeSpec` | Explicit mean/stddev normalization. |
| `preprocess.quantize` | `QuantizeSpec` | Explicit quantization override before inference. |
| `preprocess.tessellate` | `TessellateSpec` | Explicit MLA tile-layout/tessellation override. |
| `preprocess.transforms` | `list[Transform]` | Ordered expert override for the preprocess transform plan. |
| `preprocess.preset` | `NormalizePreset.None`, `ImageNet`, `COCO_YOLO` | Shortcut for common normalization recipes. |

### `preprocess.resize` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `resize.enable` | `AutoFlag` | Turns resize planning on, off, or lets planner decide. |
| `resize.width` | `int` | Target model input width; `0` lets the contract infer it. |
| `resize.height` | `int` | Target model input height; `0` lets the contract infer it. |
| `resize.mode` | `ResizeMode.Stretch`, `Letterbox`, `Crop` | Chooses stretch, aspect-preserving pad, or crop behavior. |
| `resize.pad_value` | `int` | Pixel value used for letterbox padding, commonly `114` for YOLO. |
| `resize.scaling_type` | `str` | Interpolation token such as `BILINEAR`, `NEAREST_NEIGHBOUR`, `BICUBIC`, or `INTERAREA`. |

### `preprocess.color_convert` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `color_convert.enable` | `AutoFlag` | Turns color conversion on, off, or lets planner decide. |
| `color_convert.input_format` | `PreprocessColorFormat` | Declares app-side tensor format: `RGB`, `BGR`, `GRAY8`, `NV12`, `I420`, or `Auto`. |
| `color_convert.output_format` | `PreprocessColorFormat` | Declares model-side color format, or `Auto` to use the model contract. |

### `preprocess.layout_convert` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `layout_convert.enable` | `AutoFlag` | Turns layout conversion on, off, or lets planner decide. |
| `layout_convert.perm` | `list[int]` | Axis permutation, for example `[2, 0, 1]` for HWC to CHW. |
| `layout_convert.has_perm()` | method | Returns whether a permutation was explicitly set. |

### `preprocess.normalize` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `normalize.enable` | `AutoFlag` | Turns explicit normalization on, off, or lets planner decide. |
| `normalize.mean` | `list[float]` | Per-channel mean to subtract. |
| `normalize.stddev` | `list[float]` | Per-channel divisor after mean subtraction. |
| `normalize.has_explicit_stats` | `bool` | Indicates whether explicit stats were set instead of using defaults/preset. |

### `preprocess.quantize` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `quantize.enable` | `AutoFlag` | Turns explicit quantization on, off, or lets planner decide. |
| `quantize.zero_point` | `int` | Quantization zero-point offset. |
| `quantize.scale` | `float` | Quantization scale; `0` means use model calibration/contract. |
| `quantize.output_dtype` | `str` | Target quantized dtype name, such as `int8` or `uint8`. |

### `preprocess.tessellate` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `tessellate.enable` | `AutoFlag` | Turns explicit tessellation on, off, or lets planner decide. |
| `tessellate.slice_shape` | `list[int]` | Tile shape override, usually `[H, W, C]`; leave empty unless contract requires it. |
| `tessellate.set_slice_shape(shape)` | method | Sets the tessellation slice shape from a Python list. |
| `tessellate.has_slice_shape()` | method | Returns whether a complete slice shape exists. |

### `preprocess.transforms` Element Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `transform.type` | `TransformType` | Selects which transform spec is active. |
| `transform.resize` | `ResizeSpec` | Resize spec used when `type` is `Resize`. |
| `transform.color_convert` | `ColorConvertSpec` | Color conversion spec used when `type` is `ColorConvert`. |
| `transform.layout_convert` | `LayoutConvertSpec` | Layout conversion spec used when `type` is `LayoutConvert`. |
| `transform.normalize` | `NormalizeSpec` | Normalization spec used when `type` is `Normalize`. |
| `transform.quantize` | `QuantizeSpec` | Quantization spec used when `type` is `Quantize`. |
| `transform.tessellate` | `TessellateSpec` | Tessellation spec used when `type` is `Tessellate`. |

### `verbose` Sub-Options

| Field / helper | Type / values | One-line use |
| --- | --- | --- |
| `verbose.level` | `VerbosityLevel` | Overall verbosity level such as quiet, production, or verbose. |
| `verbose.progress` | `bool` | Enables progress messages from runtime/model setup. |
| `verbose.progress_force` | `bool` | Forces progress output even when normal heuristics would suppress it. |
| `verbose.gstreamer` | `bool` | Enables GStreamer-related diagnostics. |
| `verbose.planner` | `bool` | Enables model route/preprocess planner diagnostics. |
| `verbose.graph` | `bool` | Enables graph construction diagnostics. |
| `verbose.pipeline` | `bool` | Enables generated pipeline diagnostics. |
| `verbose.inputstream` | `bool` | Enables input-stream diagnostics. |
| `verbose.tensor` | `bool` | Enables tensor diagnostics. |
| `verbose.plugins` | `bool` | Enables plugin diagnostics. |
| `VerboseOptions.quiet()` | static helper | Returns a quiet preset for notebooks or clean logs. |
| `VerboseOptions.production()` | static helper | Returns a production-oriented diagnostic preset. |
| `VerboseOptions.debug_plugins()` | static helper | Returns a plugin-debug preset. |
| `VerboseOptions.debug_all()` | static helper | Returns a broad debug preset. |

### `inference_terminal` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `inference_terminal.mla_only` | `bool` | Stops after MLA inference for internal route debugging. |
| `inference_terminal.last_stage_index` | `int` | Stops after a selected stage index. |
| `inference_terminal.last_stage_name` | `str` | Stops after a selected stage name. |
| `inference_terminal.last_plugin_id` | `str` | Stops after a selected plugin ID. |
| `inference_terminal.last_processor` | `str` | Stops after a selected processor type. |

### `processcvu` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `processcvu.pre_run_target` | `str` | Target token for pre-MLA CVU work, commonly `AUTO`, `EV74`, or `A65`. |
| `processcvu.post_run_target` | `str` | Target token for post-MLA CVU work, commonly `AUTO`, `EV74`, or `A65`. |
| `processcvu.async_` | `bool` | Enables async process-CVU submit path when eligible. |

### `processmla` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `processmla.async_` | `bool` | Enables async MLA submit/output path when eligible. |
| `processmla.output_pool_buffers` | `int` | Overrides MLA output pool size; `0` keeps runtime default. |
| `processmla.defer_output_invalidate` | `bool` | Defers producer-side output cache invalidation until CPU consumer boundary. |

### `advanced_execution` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `advanced_execution.preprocess_target` | `str` or `None` | High-level target override for model-managed preprocessing. |
| `advanced_execution.postprocess_target` | `str` or `None` | High-level target override for model-managed postprocessing. |
| `advanced_execution.preprocess_async` | `bool` or `None` | Allows preprocessing to run asynchronously when set. |
| `advanced_execution.inference_async` | `bool` or `None` | Allows MLA inference to run asynchronously when set. |
| `advanced_execution.inference_output_buffers` | `int` or `None` | Overrides inference output buffer count when measurement shows a need. |
| `advanced_execution.defer_output_cache_sync` | `bool` or `None` | Defers output cache sync for advanced memory-lifetime tuning. |
| `advanced_execution.prepared_runner` | `PreparedRunnerOptions` or `None` | Enables prepared-runner experiments through a nested object. |
| `advanced_execution.internal_queue_depth` | `int` or `None` | Sets internally inserted queue depth for jitter absorption. |

### `advanced_execution.prepared_runner` Sub-Options

| Field | Type / values | One-line use |
| --- | --- | --- |
| `prepared_runner.mode` | `str` | Prepared-runner mode token, such as dequant route experiments. |
| `prepared_runner.ring_depth` | `int` | Ring depth for prepared-runner buffering. |
| `prepared_runner.profile` | `bool` | Enables prepared-runner profiling. |
| `prepared_runner.dequant_flags` | `str` | Optional dequant optimization flags for prepared-runner experiments. |


## Public Fields

The safest way to learn the option surface is to inspect the public fields from the installed runtime. Different SDK versions can add fields, so this notebook starts with discovery instead of a hand-written list only.


In [2]:
def public_names(obj):
    out = [name for name in dir(obj) if not name.startswith("_")]
    return out, len(out)

opt = pyneat.ModelOptions()

print("ModelOptions:")
print(public_names(opt))
print("\nPreprocessOptions:")
print(public_names(opt.preprocess))
print("\nResizeSpec:")
print(public_names(opt.preprocess.resize))
print("\nColorConvertSpec:")
print(public_names(opt.preprocess.color_convert))
print("\nNormalizeSpec:")
print(public_names(opt.preprocess.normalize))
print("\nQuantizeSpec:")
print(public_names(opt.preprocess.quantize))
print("\nTessellateSpec:")
print(public_names(opt.preprocess.tessellate))


ModelOptions:
(['advanced_execution', 'boxdecode_original_height', 'boxdecode_original_width', 'boxdecode_resize_mode', 'cleanup_extracted_model_data', 'decode_type', 'decode_type_option', 'inference_terminal', 'name_suffix', 'nms_iou_threshold', 'num_classes', 'preprocess', 'processcvu', 'processmla', 'score_threshold', 'top_k', 'upstream_name', 'verbose'], 18)

PreprocessOptions:
(['color_convert', 'enable', 'input_max_depth', 'input_max_height', 'input_max_width', 'kind', 'layout_convert', 'normalize', 'preset', 'quantize', 'resize', 'tessellate', 'transforms'], 13)

ResizeSpec:
(['enable', 'height', 'mode', 'pad_value', 'scaling_type', 'width'], 6)

ColorConvertSpec:
(['enable', 'input_format', 'output_format'], 3)

NormalizeSpec:
(['enable', 'has_explicit_stats', 'mean', 'stddev'], 4)

QuantizeSpec:
(['enable', 'output_dtype', 'scale', 'zero_point'], 4)

TessellateSpec:
(['enable', 'has_slice_shape', 'set_slice_shape', 'slice_shape'], 4)


## Enum Values

Most ModelOptions fields use typed enums. Use enum members, not strings, when the API exposes an enum.


In [3]:
def public_names(obj):
    out = [name for name in dir(obj) if not name.startswith("_")]
    return out
    
for name in [
    "InputKind",
    "AutoFlag",
    "PreprocessColorFormat",
    "ResizeMode",
    "NormalizePreset",
    "TransformType",
    "VerbosityLevel",
    "BoxDecodeType",
    "BoxDecodeTypeOption",
]:
    enum_obj = getattr(pyneat, name)
    
    print(f"{name}:")
    for item in public_names(enum_obj):
        print(" -", item)
    print()


InputKind:
 - Auto
 - Image
 - Tensor

AutoFlag:
 - Auto
 - Off
 - On

PreprocessColorFormat:
 - Auto
 - BGR
 - GRAY8
 - I420
 - NV12
 - RGB

ResizeMode:
 - Crop
 - Letterbox
 - Stretch

NormalizePreset:
 - COCO_YOLO
 - ImageNet
 - None

TransformType:
 - ColorConvert
 - LayoutConvert
 - Normalize
 - Quantize
 - Resize
 - Tessellate

VerbosityLevel:
 - Production
 - Quiet
 - Verbose

BoxDecodeType:
 - Centernet
 - Detr
 - EffDet
 - RcnnStage1
 - Unspecified
 - Yolo
 - YoloV10
 - YoloV10Seg
 - YoloV26
 - YoloV26Pose
 - YoloV26Seg
 - YoloV5
 - YoloV5Seg
 - YoloV6
 - YoloV7
 - YoloV7Seg
 - YoloV8
 - YoloV8Pose
 - YoloV8Seg
 - YoloV9
 - YoloV9Seg
 - YoloX

BoxDecodeTypeOption:
 - Auto
 - GroupedByRole
 - GroupedByRoleLogit
 - GroupedByRoleProbability
 - InterleavedByHead
 - InterleavedByHeadLogit
 - InterleavedByHeadProbability
 - PackedPerHead
 - Split3Grouped
 - Split3Interleaved



In [4]:
[name for name in dir(pyneat.NormalizePreset) if not name.startswith("_")]

['COCO_YOLO', 'ImageNet', 'None']

In [5]:
[name for name in dir(pyneat.InputKind) if not name.startswith("_")]

['Auto', 'Image', 'Tensor']

In [6]:
[name for name in dir(pyneat.PreprocessColorFormat) if not name.startswith("_")]

['Auto', 'BGR', 'GRAY8', 'I420', 'NV12', 'RGB']

In [7]:
[name for name in dir(pyneat.BoxDecodeType) if not name.startswith("_")]

['Centernet',
 'Detr',
 'EffDet',
 'RcnnStage1',
 'Unspecified',
 'Yolo',
 'YoloV10',
 'YoloV10Seg',
 'YoloV26',
 'YoloV26Pose',
 'YoloV26Seg',
 'YoloV5',
 'YoloV5Seg',
 'YoloV6',
 'YoloV7',
 'YoloV7Seg',
 'YoloV8',
 'YoloV8Pose',
 'YoloV8Seg',
 'YoloV9',
 'YoloV9Seg',
 'YoloX']

In [8]:
[name for name in dir(pyneat.BoxDecodeTypeOption) if not name.startswith("_")]

['Auto',
 'GroupedByRole',
 'GroupedByRoleLogit',
 'GroupedByRoleProbability',
 'InterleavedByHead',
 'InterleavedByHeadLogit',
 'InterleavedByHeadProbability',
 'PackedPerHead',
 'Split3Grouped',
 'Split3Interleaved']

## Mental Model

`ModelOptions` has a root object and a nested `preprocess` object.

Important root fields:

| Field | Use |
| --- | --- |
| `preprocess` | Image/tensor input intent and preprocessing plan. |
| `decode_type` | Add model-managed postprocessing for detection, pose, or segmentation outputs. |
| `decode_type_option` | Select a known tensor packing variant; keep `Auto` unless you know the model head layout. |
| `score_threshold` | Drop low-confidence detection candidates. |
| `nms_iou_threshold` | Merge overlapping detection boxes with NMS. |
| `top_k` | Cap decoded detections for deterministic downstream cost. |
| `num_classes` | Set only when class count cannot be inferred reliably from the model package. |
| `name_suffix` | Make generated route element names unique when composing many models. |
| `verbose` | Control construction/planner diagnostics. |
| `cleanup_extracted_model_data` | Keep extracted model files for debugging when set to `False`. |
| `advanced_execution` | Advanced placement/async experiments after a measured baseline. |

Deprecated transition fields such as `boxdecode_original_width` and `boxdecode_original_height` can still exist, but new examples should prefer preprocess metadata for coordinate inversion.


## Preprocess Field Guide

`opt.preprocess` describes the data your application sends and what Neat may adapt before MLA inference.

| Field | Use |
| --- | --- |
| `kind` | `Image` for decoded pixels, `Tensor` for already model-shaped tensors, `Auto` for planner inference. |
| `enable` | Master switch. Use `Off` only when input already matches the model contract. |
| `input_max_width`, `input_max_height`, `input_max_depth` | Bounds for dynamic image input. Useful for video/image routes. |
| `resize` | Resize, stretch, letterbox, crop, pad value, and scaling token. |
| `color_convert` | Declare source and target color formats: RGB, BGR, GRAY8, NV12, I420, or Auto. |
| `layout_convert` | Axis order conversion such as HWC to CHW when needed. |
| `normalize` | Explicit mean/stddev normalization. |
| `preset` | Shorthand normalization preset such as ImageNet or COCO_YOLO. |
| `quantize` | Explicit quantization controls; usually leave to model contract/calibration. |
| `tessellate` | Explicit MLA tile-layout controls; usually leave to the model contract. |
| `transforms` | Ordered expert override for transform planning. Most apps should avoid this. |


## Pattern: Image Classification

Use this when your app sends RGB image tensors to an ImageNet classification model such as ResNet-50.


In [9]:
IMAGE_SIZE = 224

cls_opt = pyneat.ModelOptions()
cls_opt.preprocess.kind = pyneat.InputKind.Image  # Application input is decoded image pixels.
cls_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.RGB  # Tensor passed by the app is RGB.
cls_opt.preprocess.input_max_width = IMAGE_SIZE  # Largest width this route will accept.
cls_opt.preprocess.input_max_height = IMAGE_SIZE  # Largest height this route will accept.
cls_opt.preprocess.input_max_depth = 3  # RGB has three channels.
cls_opt.preprocess.resize.enable = pyneat.AutoFlag.On  # Let Neat resize if input is not model-sized.
cls_opt.preprocess.resize.width = IMAGE_SIZE  # Classification model input width.
cls_opt.preprocess.resize.height = IMAGE_SIZE  # Classification model input height.
cls_opt.preprocess.resize.mode = pyneat.ResizeMode.Stretch  # Common classification resize behavior.
cls_opt.preprocess.preset = pyneat.NormalizePreset.ImageNet  # ImageNet mean/stddev preset.
cls_opt.verbose = pyneat.VerboseOptions.quiet()  # Reduce notebook construction logs.

print("classification ModelOptions ready")
print("preset:", cls_opt.preprocess.preset)
print("resize:", cls_opt.preprocess.resize.width, cls_opt.preprocess.resize.height, cls_opt.preprocess.resize.mode)


classification ModelOptions ready
preset: NormalizePreset.ImageNet
resize: 224 224 ResizeMode.Stretch


## Pattern: YOLO Detection From OpenCV BGR

OpenCV `cv2.imread` and many OpenCV pipelines produce BGR images. Declare BGR input, then let Neat convert and normalize according to the model route.


In [10]:
MODEL_W = 640
MODEL_H = 640

bgr_yolo_opt = pyneat.ModelOptions()
bgr_yolo_opt.preprocess.kind = pyneat.InputKind.Image  # The app sends image pixels, not a model-shaped tensor.
bgr_yolo_opt.preprocess.color_convert.enable = pyneat.AutoFlag.On  # Enable conversion when route needs it.
bgr_yolo_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.BGR  # OpenCV image format.
bgr_yolo_opt.preprocess.color_convert.output_format = pyneat.PreprocessColorFormat.RGB  # Common YOLO model color input.
bgr_yolo_opt.preprocess.input_max_width = MODEL_W  # Bound accepted image width.
bgr_yolo_opt.preprocess.input_max_height = MODEL_H  # Bound accepted image height.
bgr_yolo_opt.preprocess.input_max_depth = 3  # BGR has three channels.
bgr_yolo_opt.preprocess.resize.enable = pyneat.AutoFlag.On  # Resize is needed when source size differs.
bgr_yolo_opt.preprocess.resize.width = MODEL_W  # YOLO model width.
bgr_yolo_opt.preprocess.resize.height = MODEL_H  # YOLO model height.
bgr_yolo_opt.preprocess.resize.mode = pyneat.ResizeMode.Letterbox  # Preserve aspect ratio for detection.
bgr_yolo_opt.preprocess.resize.pad_value = 114  # Common YOLO letterbox pad value.
bgr_yolo_opt.preprocess.resize.scaling_type = "BILINEAR"  # Interpolation token used by preproc.
bgr_yolo_opt.preprocess.preset = pyneat.NormalizePreset.COCO_YOLO  # YOLO/COCO normalization preset.

bgr_yolo_opt.decode_type = pyneat.BoxDecodeType.YoloV8  # Decode YOLOv8/YOLO11-style boxes.
bgr_yolo_opt.decode_type_option = pyneat.BoxDecodeTypeOption.Auto  # Let the route infer head packing.
bgr_yolo_opt.score_threshold = 0.25  # Drop weak candidates.
bgr_yolo_opt.nms_iou_threshold = 0.45  # NMS overlap threshold.
bgr_yolo_opt.top_k = 100  # Maximum decoded detections.
bgr_yolo_opt.num_classes = 80  # COCO class count when explicit class count is useful.
bgr_yolo_opt.verbose = pyneat.VerboseOptions.quiet()  # Keep notebook output compact.

print("BGR YOLO detection ModelOptions ready")
print("decode:", bgr_yolo_opt.decode_type)
print("thresholds:", bgr_yolo_opt.score_threshold, bgr_yolo_opt.nms_iou_threshold, bgr_yolo_opt.top_k)


BGR YOLO detection ModelOptions ready
decode: BoxDecodeType.YoloV8
thresholds: 0.25 0.44999998807907104 100


## Pattern: Real-Time NV12 Detection

Video decoder nodes often produce NV12. In a streaming app, avoid unnecessary BGR/RGB conversion in Python or C++ when the model route can consume NV12 and convert inside Neat preprocessing.


In [11]:
nv12_yolo_opt = pyneat.ModelOptions()
nv12_yolo_opt.preprocess.kind = pyneat.InputKind.Image  # Decoded video frame.
nv12_yolo_opt.preprocess.enable = pyneat.AutoFlag.On  # Let model-managed preproc run.
nv12_yolo_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.NV12  # Decoder output.
nv12_yolo_opt.preprocess.color_convert.output_format = pyneat.PreprocessColorFormat.RGB  # Model-side color contract.
nv12_yolo_opt.preprocess.resize.enable = pyneat.AutoFlag.On  # Needed for 1280x720 -> 640x640, etc.
nv12_yolo_opt.preprocess.resize.width = MODEL_W  # Target model width.
nv12_yolo_opt.preprocess.resize.height = MODEL_H  # Target model height.
nv12_yolo_opt.preprocess.resize.mode = pyneat.ResizeMode.Letterbox  # Preserve detection geometry.
nv12_yolo_opt.preprocess.resize.pad_value = 114  # YOLO pad value.
nv12_yolo_opt.preprocess.preset = pyneat.NormalizePreset.COCO_YOLO  # YOLO normalization.
nv12_yolo_opt.decode_type = pyneat.BoxDecodeType.YoloV8  # Detection decode.
nv12_yolo_opt.score_threshold = 0.25  # Confidence threshold.
nv12_yolo_opt.nms_iou_threshold = 0.45  # NMS threshold.
nv12_yolo_opt.top_k = 100  # Detection cap.
nv12_yolo_opt.num_classes = 80  # COCO class count.

print("NV12 YOLO detection ModelOptions ready")


NV12 YOLO detection ModelOptions ready


## Pattern: YOLO26 And Segmentation Variants

The preprocess setup can stay similar, but `decode_type` changes with the output contract.


In [12]:
yolo26_opt = pyneat.ModelOptions()
yolo26_opt.preprocess.kind = pyneat.InputKind.Image
yolo26_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.NV12
yolo26_opt.preprocess.preset = pyneat.NormalizePreset.COCO_YOLO
yolo26_opt.decode_type = pyneat.BoxDecodeType.YoloV26  # YOLO26 detection decode.
yolo26_opt.score_threshold = 0.25
yolo26_opt.nms_iou_threshold = 0.50
yolo26_opt.top_k = 100

seg_opt = pyneat.ModelOptions()
seg_opt.preprocess.kind = pyneat.InputKind.Image
seg_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.NV12
seg_opt.preprocess.preset = pyneat.NormalizePreset.COCO_YOLO
seg_opt.decode_type = pyneat.BoxDecodeType.YoloV8Seg  # Return boxes plus masks via decode_segmentation.
seg_opt.score_threshold = 0.25
seg_opt.nms_iou_threshold = 0.50
seg_opt.top_k = 100

print("YOLO26 decode:", yolo26_opt.decode_type)
print("YOLOv8 segmentation decode:", seg_opt.decode_type)


YOLO26 decode: BoxDecodeType.YoloV26
YOLOv8 segmentation decode: BoxDecodeType.YoloV8Seg


## Pattern: Raw Model-Shaped Tensor

Use this only when your application already creates the exact tensor shape, dtype, layout, quantization, and tessellation expected by the compiled model package. If you are sending ordinary image pixels, use `InputKind.Image` instead.


In [13]:
raw_tensor_opt = pyneat.ModelOptions()
raw_tensor_opt.preprocess.kind = pyneat.InputKind.Tensor  # Input is already model-shaped.
raw_tensor_opt.preprocess.enable = pyneat.AutoFlag.Off  # Skip model-managed preprocessing.
raw_tensor_opt.verbose = pyneat.VerboseOptions.quiet()

print("raw tensor ModelOptions ready")
print("kind:", raw_tensor_opt.preprocess.kind)
print("preprocess enabled:", raw_tensor_opt.preprocess.enable)


raw tensor ModelOptions ready
kind: InputKind.Tensor
preprocess enabled: AutoFlag.Off


## Resize Details

`ResizeMode` changes geometry and can change accuracy.

| Mode | Behavior | Common use |
| --- | --- | --- |
| `Stretch` | Scale width and height independently. | Classification models where fixed resize was used during training. |
| `Letterbox` | Preserve aspect ratio and pad. | YOLO detection/segmentation where box geometry matters. |
| `Crop` | Isotropic scale and center crop. | Classification pipelines trained with center crop behavior. |

`scaling_type` controls interpolation. Common tokens include `BILINEAR`, `NEAREST_NEIGHBOUR`, `BICUBIC`, `INTERAREA`, and `NO_SCALING`; aliases such as `INTER_AREA` may also be accepted depending on runtime.


In [14]:
resize_examples = {}

stretch = pyneat.ModelOptions()
stretch.preprocess.resize.enable = pyneat.AutoFlag.On
stretch.preprocess.resize.width = 224
stretch.preprocess.resize.height = 224
stretch.preprocess.resize.mode = pyneat.ResizeMode.Stretch
stretch.preprocess.resize.scaling_type = "INTERAREA"
resize_examples["classification_stretch"] = stretch

letterbox = pyneat.ModelOptions()
letterbox.preprocess.resize.enable = pyneat.AutoFlag.On
letterbox.preprocess.resize.width = 640
letterbox.preprocess.resize.height = 640
letterbox.preprocess.resize.mode = pyneat.ResizeMode.Letterbox
letterbox.preprocess.resize.pad_value = 114
letterbox.preprocess.resize.scaling_type = "BILINEAR"
resize_examples["detection_letterbox"] = letterbox

for name, example in resize_examples.items():
    r = example.preprocess.resize
    print(name, "->", r.width, r.height, r.mode, r.pad_value, r.scaling_type)


classification_stretch -> 224 224 ResizeMode.Stretch 114 INTERAREA
detection_letterbox -> 640 640 ResizeMode.Letterbox 114 BILINEAR


## Normalization: Preset Or Explicit Stats

Use a preset when it exactly matches the model training preprocessing. Use explicit stats when the model was trained with custom normalization.


In [15]:
preset_opt = pyneat.ModelOptions()
preset_opt.preprocess.preset = pyneat.NormalizePreset.ImageNet  # Shortcut for common ImageNet stats.

explicit_opt = pyneat.ModelOptions()
explicit_opt.preprocess.normalize.enable = pyneat.AutoFlag.On  # Turn explicit normalization on.
explicit_opt.preprocess.normalize.mean = [0.485, 0.456, 0.406]  # Per-channel mean.
explicit_opt.preprocess.normalize.stddev = [0.229, 0.224, 0.225]  # Per-channel divisor.

print("preset:", preset_opt.preprocess.preset)
print("explicit mean:", explicit_opt.preprocess.normalize.mean)
print("explicit stddev:", explicit_opt.preprocess.normalize.stddev)


preset: NormalizePreset.ImageNet
explicit mean: [0.48500001430511475, 0.4560000002384186, 0.4059999883174896]
explicit stddev: [0.2290000021457672, 0.2240000069141388, 0.22499999403953552]


## Color Conversion: Input And Output Format

`input_format` describes what your tensor contains. `output_format` describes what should be handed to the model preprocessing route. If `output_format` is `Auto`, Neat uses the model contract when it can infer it.


In [16]:
color_opt = pyneat.ModelOptions()
color_opt.preprocess.kind = pyneat.InputKind.Image
color_opt.preprocess.color_convert.enable = pyneat.AutoFlag.On
color_opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.BGR  # App-side OpenCV image.
color_opt.preprocess.color_convert.output_format = pyneat.PreprocessColorFormat.RGB  # Model-side color order.

print("input color:", color_opt.preprocess.color_convert.input_format)
print("output color:", color_opt.preprocess.color_convert.output_format)


input color: PreprocessColorFormat.BGR
output color: PreprocessColorFormat.RGB


## Detection Decode Details

Set these fields only when you want the model route to return decoded detections instead of raw output heads.

| Field | Practical rule |
| --- | --- |
| `decode_type` | Required for Neat-managed detection/pose/segmentation decode. |
| `decode_type_option` | Keep `Auto` unless the tensor packing is known. |
| `score_threshold` | Raise to remove weak detections; lower for recall. |
| `nms_iou_threshold` | Higher keeps more overlapping boxes; lower merges more aggressively. |
| `top_k` | Keep finite for predictable CPU/output cost. |
| `num_classes` | Set for custom/single-class heads when MPK metadata is not enough. |

Avoid using `boxdecode_original_width` and `boxdecode_original_height` in new code unless you are maintaining an older example. Modern coordinate inversion should use preprocess metadata created by the route.


In [17]:
def detection_decode_summary(opt):
    return {
        "decode_type": opt.decode_type,
        "decode_type_option": opt.decode_type_option,
        "score_threshold": opt.score_threshold,
        "nms_iou_threshold": opt.nms_iou_threshold,
        "top_k": opt.top_k,
        "num_classes": opt.num_classes,
    }

print(detection_decode_summary(bgr_yolo_opt))


{'decode_type': BoxDecodeType.YoloV8, 'decode_type_option': BoxDecodeTypeOption.Auto, 'score_threshold': 0.25, 'nms_iou_threshold': 0.44999998807907104, 'top_k': 100, 'num_classes': 80}


## Verbose Options

`opt.verbose` controls model construction and planner diagnostics. Use `quiet()` for clean notebooks, `production()` for normal apps, and debug presets when collecting evidence.


In [18]:
quiet_opt = pyneat.ModelOptions()
quiet_opt.verbose = pyneat.VerboseOptions.quiet()

production_opt = pyneat.ModelOptions()
production_opt.verbose = pyneat.VerboseOptions.production()

plugin_debug_opt = pyneat.ModelOptions()
plugin_debug_opt.verbose = pyneat.VerboseOptions.debug_plugins()

print("quiet verbose:", quiet_opt.verbose)
print("production verbose:", production_opt.verbose)
print("plugin debug verbose:", plugin_debug_opt.verbose)


quiet verbose: <pyneat._pyneat_core.VerboseOptions object at 0xffff09558570>
production verbose: <pyneat._pyneat_core.VerboseOptions object at 0xffff095582b0>
plugin debug verbose: <pyneat._pyneat_core.VerboseOptions object at 0xffff09558570>


## Model Construction And Inspection

Options are applied when the model is constructed. If a model package exists, you can inspect the resolved specs and preprocess plan. Keep this cell optional so the notebook remains useful even before models are downloaded.


In [19]:
MODEL_PATH = Path("../assets/models/yolo_v8n_mpk.tar.gz")
print("model path:", MODEL_PATH)
print("exists:", MODEL_PATH.exists())


model path: ../assets/models/yolo_v8n_mpk.tar.gz
exists: True


In [20]:
if MODEL_PATH.exists():
    model = pyneat.Model(str(MODEL_PATH), bgr_yolo_opt)
    print("model loaded")
    print("input specs:", model.input_specs())
    print("output specs:", model.output_specs())
    try:
        print("resolved preprocess plan:", model.resolved_preprocess_plan())
    except Exception as exc:
        print("resolved preprocess plan unavailable:", type(exc).__name__, exc)
else:
    print("Skipping model construction because the model package was not found.")


model loaded
input specs: [<pyneat._pyneat_core.TensorConstraint object at 0xffff09535960>]
output specs: [<pyneat._pyneat_core.TensorConstraint object at 0xffff09535960>]
resolved preprocess plan: ResolvedPreprocessPlan{requested.kind=Image, requested.enable=Auto, requested.preset=COCO_YOLO, requested.resize=(On, 640x640, Letterbox), requested.normalize=Auto, requested.quantize=Auto, requested.tessellate=Auto, requested.transforms=0, explicit{resize=1,color=1,layout=0,normalize=0,normalize_stats=0,quant_enable=0,quant_params=0,tess_enable=0,tess_geom=0}, effective.kind=Image, effective.enable=Auto, effective.preset=COCO_YOLO, graph_family=Preproc, graph_kernel=preproc, graph_config_path=, resolved_kind=Image, enabled=true, transforms_override=false, ingress_contracts=[{media_type=application/vnd.simaai.tensor, format=FP32, width=640, height=640, depth=3, max_width=640, max_height=640, max_depth=3}], mla={media_type=application/vnd.simaai.tensor, format=INT8, width=640, height=640, dep

## Advanced Execution Fields

`advanced_execution` is part of `ModelOptions`, but use it only after the default route is correct and measured. Set one field at a time and compare against a baseline.

Typical fields include:

| Field | Use |
| --- | --- |
| `preprocess_target` | Try a specific target for model-managed preprocessing. |
| `postprocess_target` | Try a specific target for model-managed postprocessing. |
| `preprocess_async` | Allow preprocessing to run asynchronously. |
| `inference_async` | Allow MLA inference to run asynchronously. |
| `inference_output_buffers` | Increase inference output buffering when measurement shows it is needed. |
| `defer_output_cache_sync` | Advanced memory synchronization tuning. |
| `prepared_runner` | Advanced route-runner experiment. |
| `internal_queue_depth` | Internal queue depth for jitter absorption. |


In [21]:
advanced_opt = pyneat.ModelOptions()
print("AdvancedExecutionOptions fields:")
print(public_names(advanced_opt.advanced_execution))

# Example only: do not enable these in production until the default route has a measured baseline.
advanced_opt.advanced_execution.inference_async = True
advanced_opt.advanced_execution.internal_queue_depth = 3

print("inference_async:", advanced_opt.advanced_execution.inference_async)
print("internal_queue_depth:", advanced_opt.advanced_execution.internal_queue_depth)


AdvancedExecutionOptions fields:
['defer_output_cache_sync', 'inference_async', 'inference_output_buffers', 'internal_queue_depth', 'postprocess_target', 'prepared_runner', 'preprocess_async', 'preprocess_target']
inference_async: True
internal_queue_depth: 3


## Common Recipes

### Classification

```python
opt = pyneat.ModelOptions()
opt.preprocess.kind = pyneat.InputKind.Image
opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.RGB
opt.preprocess.resize.enable = pyneat.AutoFlag.On
opt.preprocess.resize.width = 224
opt.preprocess.resize.height = 224
opt.preprocess.resize.mode = pyneat.ResizeMode.Stretch
opt.preprocess.preset = pyneat.NormalizePreset.ImageNet
```

### YOLO Detection From RTSP NV12

```python
opt = pyneat.ModelOptions()
opt.preprocess.kind = pyneat.InputKind.Image
opt.preprocess.color_convert.input_format = pyneat.PreprocessColorFormat.NV12
opt.preprocess.color_convert.output_format = pyneat.PreprocessColorFormat.RGB
opt.preprocess.resize.enable = pyneat.AutoFlag.On
opt.preprocess.resize.width = 640
opt.preprocess.resize.height = 640
opt.preprocess.resize.mode = pyneat.ResizeMode.Letterbox
opt.preprocess.resize.pad_value = 114
opt.preprocess.preset = pyneat.NormalizePreset.COCO_YOLO
opt.decode_type = pyneat.BoxDecodeType.YoloV8
opt.score_threshold = 0.25
opt.nms_iou_threshold = 0.45
opt.top_k = 100
```

### Already Model-Shaped Tensor

```python
opt = pyneat.ModelOptions()
opt.preprocess.kind = pyneat.InputKind.Tensor
opt.preprocess.enable = pyneat.AutoFlag.Off
```


## What To Remember

- Start with the smallest option set that describes the input you actually send.
- Use `InputKind.Image` for decoded pixels and `InputKind.Tensor` only for already model-shaped tensors.
- Match color format, resize mode, and normalization to the model training/compile contract.
- Use `decode_type` only when you want Neat to decode detection, pose, or segmentation outputs.
- Keep `decode_type_option` as `Auto` unless the model head packing is known.
- Avoid deprecated boxdecode original-size hints in new examples; preserve preprocess metadata instead.
- Treat `advanced_execution`, `processcvu`, and `processmla` as measurement-driven tuning, not first-run defaults.


## References

- Core tutorials: [configure_model_options.py](https://github.com/sima-neat/core/blob/main/tutorials/005_configure_model_options/configure_model_options.py), [preprocess_images.py](https://github.com/sima-neat/core/blob/main/tutorials/006_preprocess_images/preprocess_images.py), [read_detection_boxes.py](https://github.com/sima-neat/core/blob/main/tutorials/007_read_detection_boxes/read_detection_boxes.py).
- Core docs: [development-workflow/model.mdx](https://github.com/sima-neat/core/blob/main/docs/develop-apps/development-workflow/model.mdx) and [reference/nodes/preproc.mdx](https://github.com/sima-neat/core/blob/main/docs/reference/nodes/preproc.mdx).
